<a href="https://www.kaggle.com/code/chaudhrysuleman/medgemma-1-5-4b-it-leukemia-lora-validation?scriptVersionId=297695828" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# Install required packages
!pip install -q transformers>=4.45.0
!pip install -q peft>=0.13.0
!pip install -q accelerate>=0.34.0
!pip install -q bitsandbytes>=0.44.0
!pip install -q datasets>=3.0.0
!pip install -q pillow
!pip install -q tqdm
!pip install -q huggingface_hub

print("✅ All packages installed!")

✅ All packages installed!


In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

login(token=secret_value_0)  # Replace with your token

print("✅ Logged in to Hugging Face!")

✅ Logged in to Hugging Face!


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel, PeftConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
adapter_repo = "chaudhrysuleman/medgemma-1.5-4b-it-leukemia-lora"

# 1) Load adapter config to get the base model name
peft_cfg = PeftConfig.from_pretrained(adapter_repo)

# 2) Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    peft_cfg.base_model_name_or_path,
    torch_dtype="auto"
).to(device)

# 3) Load the LoRA adapter on top (IMPORTANT: is_trainable=True)
model = PeftModel.from_pretrained(base_model, adapter_repo, is_trainable=True).to(device)
model.train()

processor = AutoProcessor.from_pretrained(peft_cfg.base_model_name_or_path)

# Check trainables
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Trainable params:", trainable)
print("Total params    :", total)


2026-02-14 14:02:16.703504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771077736.890619      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771077736.942557      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771077737.377563      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771077737.377601      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1771077737.377604      24 computation_placer.cc:177] computation placer alr

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Trainable params: 23797760
Total params    : 4323877232


In [4]:
from PIL import Image
import torch
import torch.nn.functional as F

def predict_image(image_path, threshold_leukemia=0.35):
    """
    Predict Normal vs Leukemia + probability score.
    threshold_leukemia: tune for higher leukemia recall (lower threshold => more leukemia)
    """
    image = Image.open(image_path).convert("RGB")

    user_text = (
        "Classify this blood cell microscopy image.\n"
        "Answer with exactly ONE word: Normal or Leukemia.\n"
        "Answer:"
    )
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": user_text}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

    @torch.no_grad()
    def score(candidate):
        # prompt-only
        prompt_inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)

        # prompt+candidate
        full_text = prompt + candidate
        full_inputs = processor(images=image, text=full_text, return_tensors="pt").to(model.device)

        outputs = model(**full_inputs)
        logits = outputs.logits
        input_ids = full_inputs["input_ids"]

        prompt_len = prompt_inputs["input_ids"].shape[1]
        cand_start = prompt_len
        cand_end = input_ids.shape[1]

        if cand_end <= cand_start:
            return torch.tensor(-1e9, device=model.device, dtype=torch.float32)

        total = torch.tensor(0.0, device=model.device, dtype=torch.float32)

        for i in range(cand_start, cand_end):
            token_id = input_ids[0, i]
            prev_logits = logits[0, i - 1].to(torch.float32)  # ✅ cast to float32
            total = total + F.log_softmax(prev_logits, dim=-1)[token_id]

        return total  # float32 scalar

    s_normal = score(" Normal")
    s_leuk   = score(" Leukemia")

    # ✅ keep everything in torch, cast to float32
    scores = torch.stack([s_normal, s_leuk]).to(torch.float32)
    probs = torch.softmax(scores, dim=0)

    p_normal = probs[0].item()
    p_leuk   = probs[1].item()

    # ✅ thresholded decision (better for recall)
    pred = "Leukemia" if p_leuk >= threshold_leukemia else "Normal"
    score_out = p_leuk if pred == "Leukemia" else p_normal

    return {"prediction": pred, "p_normal": p_normal, "p_leukemia": p_leuk, "score": score_out}


In [5]:
import kagglehub

path = kagglehub.dataset_download("andrewmvd/leukemia-classification")
print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/andrewmvd/leukemia-classification


In [6]:
from pathlib import Path

BASE = Path(path) / "C-NMC_Leukemia" / "validation_data"
print("📂 validation_data contents:", [p.name for p in BASE.iterdir()])
csv_file = BASE / "C-NMC_test_prelim_phase_data_labels.csv"
print("cvs", csv_file)

# Find all images recursively under validation_data
all_imgs = []
for ext in ("*.bmp", "*.jpg", "*.png", "*.jpeg"):
    all_imgs += list(BASE.rglob(ext))

print("✅ Total images found under validation_data:", len(all_imgs))



📂 validation_data contents: ['C-NMC_test_prelim_phase_data_labels.csv', 'C-NMC_test_prelim_phase_data']
cvs /kaggle/input/datasets/andrewmvd/leukemia-classification/C-NMC_Leukemia/validation_data/C-NMC_test_prelim_phase_data_labels.csv
✅ Total images found under validation_data: 1867


In [7]:
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1) Load CSV
# csv_path = Path("/content/C-NMC_test_prelim_phase_data_labels.csv")
df = pd.read_csv(csv_file)

print("✅ CSV loaded:", csv_file)
print("Columns:", df.columns.tolist())
print(df.head())

# 2) Set these two lines to match your CSV column names
FILE_COL  = "new_names"   # <-- change if your column name is different
LABEL_COL = "labels"      # <-- change if your column name is different

# 3) Find the folder that contains the images (search recursively)
VAL_ROOT = Path(path) / "C-NMC_Leukemia" / "validation_data"
all_imgs = []
for ext in ("*.bmp", "*.jpg", "*.jpeg", "*.png"):
    all_imgs += list(VAL_ROOT.rglob(ext))

img_map = {p.name: str(p) for p in all_imgs}
print("✅ Total images found:", len(all_imgs))

# 4) Attach full file paths to each CSV row
df["img_path"] = df[FILE_COL].apply(lambda x: img_map.get(Path(str(x)).name, None))
df = df[df["img_path"].notna()].copy()

print("✅ Matched images:", len(df))

# 5) Run predictions and compare
y_true = df[LABEL_COL].astype(int).tolist()
y_pred = []
scores = []

threshold = 0.35  # lower => more leukemia predicted (higher recall)

for img_path in df["img_path"].tolist():
    out = predict_image(img_path, threshold_leukemia=threshold)

    # Convert model output to 0/1
    pred = 1 if out["prediction"].lower().startswith("leuk") else 0

    y_pred.append(pred)
    scores.append(out["score"])

# 6) Metrics
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
rec  = recall_score(y_true, y_pred, pos_label=1, zero_division=0)   # Leukemia recall
f1   = f1_score(y_true, y_pred, pos_label=1, zero_division=0)

cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

spec = tn / (tn + fp) if (tn + fp) else 0.0

print("\n✅ RESULTS (CSV label vs Model prediction)")
print(f"Threshold     : {threshold}")
print(f"Accuracy      : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall (Cancer): {rec:.4f}")
print(f"Specificity   : {spec:.4f}")
print(f"F1-score      : {f1:.4f}")
print("\nConfusion Matrix [[TN FP],[FN TP]]")
print(cm)

# 7) Save detailed comparison per image
df["y_true"] = y_true
df["y_pred"] = y_pred
df["score"]  = scores
df["correct"] = (df["y_true"] == df["y_pred"]).astype(int)

out_csv = "/content/model_vs_csv_labels.csv"
df.to_csv(out_csv, index=False)
print("\n✅ Saved per-image results to:", out_csv)
print(df[[FILE_COL, "y_true", "y_pred", "score", "correct"]].head(10))


✅ CSV loaded: /kaggle/input/datasets/andrewmvd/leukemia-classification/C-NMC_Leukemia/validation_data/C-NMC_test_prelim_phase_data_labels.csv
Columns: ['Patient_ID', 'new_names', 'labels']
             Patient_ID new_names  labels
0   UID_57_29_1_all.bmp     1.bmp       1
1   UID_57_22_2_all.bmp     2.bmp       1
2   UID_57_31_3_all.bmp     3.bmp       1
3  UID_H49_35_1_hem.bmp     4.bmp       0
4   UID_58_6_13_all.bmp     5.bmp       1
✅ Total images found: 1867
✅ Matched images: 1867

✅ RESULTS (CSV label vs Model prediction)
Threshold     : 0.35
Accuracy      : 0.7734
Precision     : 0.8373
Recall (Cancer): 0.8105
Specificity   : 0.7037
F1-score      : 0.8237

Confusion Matrix [[TN FP],[FN TP]]
[[456 192]
 [231 988]]

✅ Saved per-image results to: /content/model_vs_csv_labels.csv
  new_names  y_true  y_pred     score  correct
0     1.bmp       1       1  0.992879        1
1     2.bmp       1       1  0.999643        1
2     3.bmp       1       1  0.997817        1
3     4.bmp       